# Task 4 — Netflix Release Year Trend Analysis
Yearly title counts, Movies vs TV Shows, growth/decline, visualizations and reporting. Run all cells in order from the project root.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import darsh as da

## Load the cleaned Netflix catalog

In [ ]:
df = pd.read_csv("data/Cleaned_Netflix.csv")
print("Raw shape:", df.shape)
df.head()

## Clean and deduplicate release-year records

In [ ]:
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
df = df.dropna(subset=["show_id", "type", "release_year"]).copy()
df["release_year"] = df["release_year"].astype(int)
df = df.drop_duplicates(subset=["show_id"])
print("Clean shape:", df.shape)
df[["show_id", "title", "type", "release_year"]].head()

## Yearly content releases

In [ ]:
yearly_count = df.groupby("release_year")["show_id"].nunique().sort_index()
yearly_count.tail(15)

## Yearly Movie / TV Show comparison

In [ ]:
yearly_type = df.groupby(["release_year", "type"])["show_id"].nunique().reset_index(name="total_titles")
yearly_comparison = yearly_type.pivot(index="release_year", columns="type", values="total_titles").fillna(0)
yearly_type.tail(15)

## Growth calculation
YoY growth = (current year − previous year) / previous year × 100. Years with zero prior count have undefined growth.

In [ ]:
yearly_full = yearly_count.reindex(range(int(yearly_count.index.min()), int(yearly_count.index.max()) + 1), fill_value=0)
growth_df = yearly_full.reset_index()
growth_df.columns = ["release_year", "total_titles"]
growth_df["growth_percent"] = growth_df["total_titles"].pct_change(fill_method=None).mul(100)
growth_df.loc[growth_df["total_titles"].shift(1).eq(0), "growth_percent"] = float("nan")
growth_df.tail(15)

## Visualize yearly title counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly_full.index, yearly_full.values, marker="o", markersize=3, color="#E50914")
ax.set(title="Netflix Titles by Original Release Year", xlabel="Release Year", ylabel="Distinct Titles")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Visualize Movies vs TV Shows

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
yearly_comparison.plot(ax=ax, marker="o", markersize=3)
ax.set(title="Movies vs TV Shows by Original Release Year", xlabel="Release Year", ylabel="Distinct Titles")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Visualize growth and decline

In [ ]:
valid_growth = growth_df.dropna(subset=["growth_percent"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(valid_growth["release_year"], valid_growth["growth_percent"], color=["green" if v >= 0 else "red" for v in valid_growth["growth_percent"]])
ax.axhline(0, color="black", linewidth=1)
ax.set(title="Year-over-Year Growth and Decline", xlabel="Release Year", ylabel="Growth (%)")
plt.tight_layout()
plt.show()

## Darsh 3D scatter chart
0 = Movie, 1 = TV Show. The display depends on the installed Darsh implementation.

In [ ]:
df_3d = yearly_type.copy()
df_3d["type_code"] = df_3d["type"].map({"Movie": 0, "TV Show": 1})
chart = da.scatter_3d(df_3d, x="release_year", y="type_code", z="total_titles")
if chart is not None:
    try:
        from IPython.display import display
        display(chart)
    except Exception:
        print(chart)

## Business insights

In [ ]:
print("Distinct titles:", df["show_id"].nunique())
print("Movies:", df.loc[df["type"].eq("Movie"), "show_id"].nunique())
print("TV Shows:", df.loc[df["type"].eq("TV Show"), "show_id"].nunique())
print("Peak release year:", int(yearly_count.idxmax()))
print("Titles in peak year:", int(yearly_count.max()))
print("Negative-growth years:", int((valid_growth["growth_percent"] < 0).sum()))
print("Top release years:")
print(yearly_count.sort_values(ascending=False).head(10))

## Export CSV reports

In [ ]:
Path("data").mkdir(exist_ok=True)
yearly_type.to_csv("data/Yearly_Content_Analysis.csv", index=False)
growth_df.to_csv("data/Yearly_Growth_Analysis.csv", index=False)
print("Saved both reports to data/")

## Interpretation and limitations
These counts reflect original release years for titles in the Netflix dataset, **not** Netflix-produced content or the year a title joined Netflix. Catalog coverage and partial years can distort apparent growth or decline.